# Clean Reward Model Training

**Authors:** Iris Nguyen, Matthew Gustafson

## Reward Model Design
Our reward model will be our base language model (Gemma2-2b) with a small scalar head, finetuned with Bradley-Terry pairwise loss to prefer truthful responses.

In [1]:
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import login

In [2]:
# Gated model: Login with a HF token with gated access permission
login()
MODEL_ID = "google/gemma-2-2b"

In [3]:
class PreferenceDataset(Dataset):
    """
    rows: list of dicts with keys:
      - prompt
      - chosen
      - rejected
    """
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        return self.rows[idx]


def format_pair(prompt: str, response: str) -> str:
    return f"Prompt:\n{prompt}\n\nResponse:\n{response}"


def load_csv_to_rows(path):
    df = pd.read_csv(path)
    df = df.drop(['template_type','source'], axis=1)

    # Optional: ensure expected columns exist
    required_cols = {"prompt", "chosen", "rejected"}
    if not required_cols.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required_cols}")

    # Convert to list of dicts
    rows = df.to_dict(orient="records")
    return rows

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def collate_fn(batch, max_length=256):
    chosen_texts = [format_pair(x["prompt"], x["chosen"]) for x in batch]
    rejected_texts = [format_pair(x["prompt"], x["rejected"]) for x in batch]

    chosen = tokenizer(
        chosen_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    rejected = tokenizer(
        rejected_texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    return chosen, rejected

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=1,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def bt_loss(chosen_scores, rejected_scores):
    """
    Bradley-Terry loss:
      -log(sigmoid(s_chosen - s_rejected))
    """
    return F.softplus(-(chosen_scores - rejected_scores)).mean()

In [ ]:
rows = load_csv_to_rows("datasets/tqa_train.csv")

dataset = PreferenceDataset(rows)
loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

optimizer = AdamW(model.parameters(), lr=1e-5)

device = next(model.parameters()).device
model.train()

for epoch in range(3):
    for chosen_inputs, rejected_inputs in loader:
        chosen_inputs = {k: v.to(device) for k, v in chosen_inputs.items()}
        rejected_inputs = {k: v.to(device) for k, v in rejected_inputs.items()}

        optimizer.zero_grad(set_to_none=True)

        chosen_logits = model(**chosen_inputs).logits.squeeze(-1)
        rejected_logits = model(**rejected_inputs).logits.squeeze(-1)

        loss = bt_loss(chosen_logits, rejected_logits)
        loss.backward()
        optimizer.step()

    print(f"epoch {epoch} loss={loss.item():.4f}")